# 第六章 多无人机Agent协同

# 6.1 多Agent系统简介

> **AirSim 配置**：本节为理论介绍，无需启动模拟器。

## 6.1.1 从单机到集群

单个无人机的能力终究有限，而由多个无人机组成的集群或"蜂群"则能产生力量倍增的效应。多无人机系统的出现，源于单一平台在覆盖范围、任务效率和系统鲁棒性方面的固有局限性。协同作业的需求推动了从单机智能到集群智能的演进。

![drone_cluster.png](img/drone_cluster.png)

### 基于大模型Agent的集群协作

本章将基于大型语言模型（LLM）的智能体（Agent）作为多无人机系统的认知核心。我们将通过两个动手实验来验证两种核心协作模式——您将在 AirSim 模拟器中亲手实现它们。

多智能体系统必须解决的根本性挑战：

- **任务分配与分解**：如何将"搜索公园内的失踪者"分解为"无人机1搜索A区"、"无人机2搜索B区"？
- **通信**：智能体之间需要共享哪些信息？如何高效传递？
- **决策**：中心化统一决策，还是各自独立判断？
- **冲突消解**：如何避免无人机之间的碰撞？

## 两种核心协同模式

![drone_level.jpg](img/drone_level.jpg)

### 模式一：中心化协同（Centralized）

**一个"指挥官"统一决策，所有无人机听从指挥。**

```
人类操作员 → [指挥官LLM] → 任务分解 → 分别指挥Drone1、Drone2...
```

- **优点**：全局最优、逻辑简单、容易实现
- **缺点**：指挥官是单点故障、通信延迟敏感、扩展性有限
- **适用场景**：任务明确、通信可靠、规模较小的集群

### 模式二：分布式协同（Distributed）

**每架无人机有自己的"大脑"，通过共享信息协调行动。**

```
Drone1 [自己的LLM] ←→ 消息板 ←→ [自己的LLM] Drone2
```

- **优点**：无单点故障、可扩展、对通信中断鲁棒
- **缺点**：难以保证全局最优、需要协商机制
- **适用场景**：动态环境、大规模集群、通信不可靠的场景

| 对比项 | 中心化 | 分布式 |
|--------|--------|--------|
| 决策方式 | 一个LLM统一规划 | 每架无人机各自决策 |
| 通信模式 | 指挥官→工作者（单向） | 消息板（双向共享） |
| 容错性 | 低（指挥官故障则全停） | 高（单机故障不影响其他） |
| 实现复杂度 | 简单 | 中等 |
| 全局最优 | 容易实现 | 较难保证 |

## 主流多Agent框架简介

业界有多种成熟的多Agent框架，这里简要介绍三个代表性的：

| 框架 | 核心理念 | 特点 |
|------|----------|------|
| **LangGraph** | 状态驱动的图 | 确定性控制流、生产级可靠性 |
| **AutoGen** | 对话驱动 | 灵活的群聊机制、动态协作 |
| **CrewAI** | 角色扮演 | 直观的团队协作、低学习曲线 |

![agent_workflow.avif](img/agent_workflow.avif)

这些框架虽然功能强大，但对于理解多Agent协同的**核心原理**来说，引入了过多的抽象层。

### 本章方案

为了让学生能够**一行行看懂代码、真正理解原理**，本章采用最简方案：

- **不使用任何Agent框架**
- **纯 Python + OpenAI SDK + AirSim**
- 用 ~50 行代码实现中心化协同
- 用 ~70 行代码实现分布式协同

理解了原理之后，再去学习 LangGraph 等框架，就会有深刻的体会。